# 2D wave-packet scattering

*quantum · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/quantum/scattering_2d)

The time-dependent Schrödinger equation in two dimensions, solved with split-step Fourier on an $(N, N)$ grid. A Gaussian wave packet hits a wall with two slits (or a disk-shaped obstacle); torch.fft.fft2 switches between $x$-space, where the potential is diagonal, and $k$-space, where the kinetic energy is diagonal. A detector screen behind the barrier records the time-integrated probability: an interference pattern built from a single particle's wavefunction.

**Vector form**

$$i\,\partial_t\psi = \Big(-\tfrac12\,\nabla^2 + V(x, y)\Big)\psi,\qquad \psi(t+\Delta t) \approx M\,e^{-iV\Delta t/2}\,\mathcal F^{-1} e^{-i|\mathbf k|^2\Delta t/2}\,\mathcal F\,e^{-iV\Delta t/2}\,\psi(t)$$

**Index (tensor) form**

$$\psi_{xy} \leftarrow P_{xy}\psi_{xy},\quad \tilde\psi_{pq} = F_{px}F_{qy}\,\psi_{xy},\quad \tilde\psi_{pq} \leftarrow K_{pq}\tilde\psi_{pq},\quad \psi_{xy} = F^{-1}_{xp}F^{-1}_{yq}\,\tilde\psi_{pq},\quad \psi_{xy} \leftarrow M_{xy}P_{xy}\psi_{xy}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\psi_{xy}$ | `psi` | (N, N) complex | wavefunction on the periodic 2D grid (first index $x$, second $y$) |
| $V_{xy}$ | `double_slit(X, Y, ...)` | (N, N) | barrier: a high wall at $x \approx 0$ with two openings, or a disk |
| $P_{xy} = e^{-iV_{xy}\Delta t/2}$ | `potential_phase(V, dt)` | (N, N) complex | half-step potential operator, diagonal in $x$-space |
| $K_{pq} = e^{-i(k_p^2+k_q^2)\Delta t/2}$ | `kinetic_phase(KX, KY, dt)` | (N, N) complex | full-step kinetic operator, diagonal in $k$-space |
| $F_{px}F_{qy}\psi_{xy}$ | `torch.fft.fft2(psi)` | (N, N) complex | $x$ and $y$ both summed: a 1D Fourier transform along each axis |
| $M_{xy}$ | `absorbing_mask(X, Y, L, width)` | (N, N) | smooth damping near the edges, so waves leave instead of wrapping around |
| $I_y$ | `screen += \|psi[ix_screen, :]\|² dt` | (N,) | time-integrated probability on the detector line $x = x_\text{screen}$ |

Units $\hbar = m = 1$, packet energy $E \approx k_0^2/2$. Without the mask $M$ every factor is a pure phase, so each step is exactly unitary and the norm stays 1; the mask deliberately removes probability that reaches the edges (otherwise the periodic FFT grid would bring it back on the opposite side), so the printed norm drops at late times. For two slits of separation $d$ the fringes on a screen a distance $D$ away are spaced by about $\lambda D/d$ with $\lambda = 2\pi/k_0$ (the screen is not in the far field, so this is only approximate). Choosing the disk shows a shadow with a bright Poisson spot on the axis behind it. The image colour uses a square-root scale normalised to each frame, so the weak scattered waves stay visible; the barrier is drawn in grey.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/quantum/scattering_2d/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors

LENGTH = 80.0  # box side
X0, X_SCREEN, V0 = -15.0, 18.0, 40.0  # packet start, detector line, barrier height


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    k0 = params["k0"]
    X, Y, KX, KY, dx = tensors.grid(params["n_grid"], LENGTH, dev)
    dA = dx * dx
    if params["barrier"] == "disk":
        V = tensors.disk(X, Y, V0, params["slit_sep"])
    else:
        V = tensors.double_slit(X, Y, V0, params["slit_sep"], params["slit_width"])
    M = tensors.absorbing_mask(X, Y, LENGTH, width=8.0)
    psi0 = tensors.gaussian_packet(X, Y, X0, sx=3.0, sy=5.0, k0=k0, dA=dA)

    column = int(((X[:, 0] - X_SCREEN).abs()).argmin())  # grid index of the detector line
    t_max = (X_SCREEN - X0 + 12.0) / k0  # long enough for the packet to pass the screen
    result = solve.solve(psi0, V, KX, KY, dA, t_max, M=M, screen_column=column)

    print(f"packet energy E = {k0**2 / 2:.3f}, barrier V0 = {V0}, wavelength = {2 * math.pi / k0:.3f}")
    if params["barrier"] == "double_slit":
        D, d = X_SCREEN, params["slit_sep"]
        print(f"expected fringe spacing ~ lambda D / d = {2 * math.pi / k0 * D / d:.2f}")
    print(f"norm: start {result['norm'][0]:.6f}   end {result['norm'][-1]:.6f}  (absorbed at the edges)")
    print(f"probability through the detector line: {result['screen'][-1].sum() * dx:.4f}")
    return plot.animate(X, Y, V, result, X_SCREEN, out_dir / "scattering_2d.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/quantum/scattering_2d/equation.py
r"""Physics → tensor translation.

Time-dependent Schrödinger equation in 2D (units ħ = m = 1):

    i \partial_t \psi = \Big(-\tfrac12 (\partial_x^2 + \partial_y^2) + V(x, y)\Big) \psi.

Exactly as in 1D, Strang splitting turns one time step into three diagonal operators:

    potential (x-space):  \psi_{xy} \leftarrow P_{xy} \psi_{xy},          P_{xy} = e^{-i V_{xy} \Delta t/2}
    kinetic (k-space):    \tilde\psi_{pq} \leftarrow K_{pq} \tilde\psi_{pq},  K_{pq} = e^{-i (k_p^2 + k_q^2) \Delta t/2}

No index is summed in either line: a diagonal operator D_{xy,x'y'} = d_{xy} \delta_{xx'} \delta_{yy'} is just an
elementwise multiply. The 2D Fourier transform sums over both grid indices,

    \tilde\psi_{pq} = F_{px} F_{qy} \psi_{xy}          (x and y summed: torch.fft.fft2)

and factorises into one 1D transform per axis. Finally an absorbing mask M_{xy} \le 1 damps ψ near the
grid edges after every step.
"""

import torch


def potential_phase(V: torch.Tensor, dt: float) -> torch.Tensor:
    r"""P_{xy} = e^{-i V_{xy} \Delta t/2}.   V: (N, N) → (N, N) complex"""
    return torch.exp(-0.5j * V * dt)


def kinetic_phase(KX: torch.Tensor, KY: torch.Tensor, dt: float) -> torch.Tensor:
    r"""K_{pq} = e^{-i (k_p^2 + k_q^2) \Delta t/2}.   KX, KY: (N, N) → (N, N) complex"""
    return torch.exp(-0.5j * (KX**2 + KY**2) * dt)


def split_step(psi: torch.Tensor, P: torch.Tensor, K: torch.Tensor, M: torch.Tensor) -> torch.Tensor:
    r"""One Strang step  \psi \to M P F^{-1} K F P \psi.   All tensors (N, N)."""
    psi = P * psi  # half potential step, diagonal in x-space
    psi_k = torch.fft.fft2(psi)  # (x, y) → (k_x, k_y)
    psi_k = K * psi_k  # full kinetic step, diagonal in k-space
    psi = torch.fft.ifft2(psi_k)  # back to (x, y)
    return M * (P * psi)  # second half potential step, then absorb at the edges


def probability(psi: torch.Tensor, dA: float) -> torch.Tensor:
    r"""\int |\psi|^2 dA \approx \psi^*_{xy} \psi_{xy} dA  (x and y summed).   (..., N, N) → (...)"""
    return torch.einsum("...xy,...xy->...", psi.conj(), psi).real * dA


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/quantum/scattering_2d/tensors.py
"""Build the concrete tensors: the 2D grid and its wavenumbers, the barrier, the absorbing mask and the packet."""

import math

import torch


def grid(n: int, length: float, device: torch.device) -> tuple[torch.Tensor, ...]:
    """Periodic (N, N) grid over [−L/2, L/2)². Returns (X, Y, KX, KY, dx); first index is x, second is y."""
    dx = length / n
    s = -length / 2 + dx * torch.arange(n, dtype=torch.float64, device=device)  # (N,)
    k = 2 * math.pi * torch.fft.fftfreq(n, d=dx, dtype=torch.float64, device=device)  # (N,) in FFT order
    X, Y = torch.meshgrid(s, s, indexing="ij")  # (N, N) each
    KX, KY = torch.meshgrid(k, k, indexing="ij")  # (N, N) each
    return X, Y, KX, KY, dx


def double_slit(X: torch.Tensor, Y: torch.Tensor, V0: float, sep: float, width: float, thick: float = 1.5):
    """V_{xy} = V0 inside a wall |x| < thick/2, except two openings |y ∓ sep/2| < width/2. Shape (N, N)."""
    wall = X.abs() < thick / 2
    slits = ((Y - sep / 2).abs() < width / 2) | ((Y + sep / 2).abs() < width / 2)
    return torch.where(wall & ~slits, V0, 0.0)


def disk(X: torch.Tensor, Y: torch.Tensor, V0: float, diameter: float) -> torch.Tensor:
    """V_{xy} = V0 inside a disk of the given diameter centred at the origin. Shape (N, N)."""
    return torch.where(X**2 + Y**2 < (diameter / 2) ** 2, V0, 0.0)


def absorbing_mask(X: torch.Tensor, Y: torch.Tensor, length: float, width: float) -> torch.Tensor:
    """M_{xy} = m(x) m(y): 1 in the middle, falling smoothly to 0 over `width` at every edge. Shape (N, N)."""

    def edge(s: torch.Tensor) -> torch.Tensor:
        depth = ((s.abs() - (length / 2 - width)) / width).clamp(0, 1)  # 0 inside, 1 at the edge
        return torch.cos(0.5 * math.pi * depth) ** 0.125  # gentle ramp: little reflection

    return edge(X) * edge(Y)


def gaussian_packet(X: torch.Tensor, Y: torch.Tensor, x0: float, sx: float, sy: float, k0: float, dA: float):
    """ψ_{xy} ∝ exp(−(x−x0)²/(4sx²) − y²/(4sy²) + i k0 x), moving along +x, Σ|ψ|² dA = 1. Shape (N, N) complex."""
    psi = torch.exp(-((X - x0) ** 2) / (4 * sx**2) - Y**2 / (4 * sy**2) + 1j * k0 * X)
    return psi / torch.sqrt((psi.abs() ** 2).sum() * dA)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/quantum/scattering_2d/solve.py
"""March the 2D packet with split-step Fourier, recording |ψ|² snapshots and the detector screen."""

import torch

from .equation import kinetic_phase, potential_phase, probability, split_step


def solve(
    psi0: torch.Tensor,
    V: torch.Tensor,
    KX: torch.Tensor,
    KY: torch.Tensor,
    dA: float,
    t_max: float,
    M: torch.Tensor | None = None,
    screen_column: int = 0,
    dt: float = 0.02,
    n_saved: int = 120,
) -> dict[str, torch.Tensor]:
    """Return a dict with times (T,), prob (T, N, N) = |ψ|², norm (T,), screen (T, N) and the final psi.

    screen[t, y] = ∫_0^t |ψ(x_screen, y)|² dt': the probability that has crossed the detector line so far.
    M = None means no absorber (every step is then exactly unitary).
    """
    n_steps = int(round(t_max / dt))
    save_every = max(1, n_steps // n_saved)
    P = potential_phase(V, dt)  # (N, N)
    K = kinetic_phase(KX, KY, dt)  # (N, N)
    M = torch.ones_like(V) if M is None else M

    psi = psi0
    screen = torch.zeros(psi0.shape[1], dtype=torch.float64, device=psi0.device)  # (N,) along y
    times, probs, norms, screens = [0.0], [(psi0.abs() ** 2).float()], [probability(psi0, dA)], [screen.clone()]
    for step in range(1, n_steps + 1):
        psi = split_step(psi, P, K, M)
        screen += psi[screen_column].abs() ** 2 * dt  # accumulate the column x = x_screen
        if step % save_every == 0:
            times.append(step * dt)
            probs.append((psi.abs() ** 2).float())  # float32 snapshots keep memory small
            norms.append(probability(psi, dA))
            screens.append(screen.clone())
    return {
        "times": torch.tensor(times),
        "prob": torch.stack(probs),
        "norm": torch.stack(norms),
        "screen": torch.stack(screens),
        "psi": psi,
    }


def mean_position(prob: torch.Tensor, X: torch.Tensor, Y: torch.Tensor) -> torch.Tensor:
    """(⟨x⟩, ⟨y⟩) of a probability density |ψ|² on the grid: (N, N) → (2,)."""
    weights = prob / prob.sum()
    return torch.stack([(weights * X).sum(), (weights * Y).sum()])


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/quantum/scattering_2d/plot.py
"""Animate |ψ|² in the plane (left) and the probability collected on the detector screen (right)."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import PowerNorm

from problems._common.plotting import ACCENT, frame_indices, save_gif


def animate(X: torch.Tensor, Y: torch.Tensor, V: torch.Tensor, result: dict, x_screen: float, path: Path) -> Path:
    xs, ys = X[:, 0].cpu().numpy(), Y[0, :].cpu().numpy()
    prob = result["prob"].cpu().numpy()  # (T, Nx, Ny)
    screen = result["screen"].cpu().numpy()  # (T, Ny)
    t = result["times"].cpu().numpy()
    extent = (xs[0], xs[-1], ys[0], ys[-1])

    fig, (ax, ax_s) = plt.subplots(1, 2, figsize=(7, 4), gridspec_kw={"width_ratios": [2.2, 1]}, sharey=True)
    # imshow wants rows = vertical axis, so transpose (x, y) → (y, x)
    image = ax.imshow(prob[0].T, origin="lower", extent=extent, cmap="inferno", interpolation="bilinear")
    barrier = np.ma.masked_where(V.cpu().numpy().T == 0, np.ones_like(prob[0].T))
    ax.imshow(barrier, origin="lower", extent=extent, cmap="Greys", vmin=0, vmax=1.4, alpha=0.9)
    ax.axvline(x_screen, color=ACCENT[5], lw=1, ls="--")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.grid(False)
    clock = ax.text(0.02, 0.96, "", transform=ax.transAxes, va="top", color="white", family="monospace")

    (profile,) = ax_s.plot([], [], color=ACCENT[5], lw=1.5)
    fill = [ax_s.fill_betweenx(ys, 0 * ys, color=ACCENT[5], alpha=0.25)]
    ax_s.set_xlim(0, 1.1)
    ax_s.set_xlabel("screen counts (scaled)")
    ax_s.set_title("detector at dashed line", fontsize=9)

    def update(step: int):
        image.set_data(prob[step].T)
        image.set_norm(PowerNorm(0.5, vmin=0, vmax=prob[step].max()))  # √ scale, normalised per frame
        counts = screen[step] / max(screen[-1].max(), 1e-30)  # (Ny,) scaled to the final maximum
        profile.set_data(counts, ys)
        fill[0].remove()
        fill[0] = ax_s.fill_betweenx(ys, counts, color=ACCENT[5], alpha=0.25)
        clock.set_text(f"t = {t[step]:5.2f}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 80), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'barrier': 'double_slit',
    'k0': 3.0,
    'slit_sep': 6.0,
    'slit_width': 1.5,
    'n_grid': 256
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.quantum.scattering_2d")]:
    del sys.modules[name]
main = importlib.import_module("problems.quantum.scattering_2d.main")
problem_dir = Path("problems/quantum/scattering_2d")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
